# CARLA + YOLO Live Detection

CARLA 0.9.16 and YOLO11s experiment, The camera frame goes directly from the CARLA callback into YOLO inference.


## Imports and Connection

Connect to the running CARLA server and check the Python, CUDA, and map setup.


In [ ]:
import random
import time
import os
import threading
import math

import numpy as np
import cv2

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from ultralytics import YOLO
from collections import defaultdict

import carla

client = carla.Client('localhost', 2000)
client.set_timeout(10.0)

world = client.get_world()

blueprint_library = world.get_blueprint_library()

print(f"Connected to CARLA")
print(f"Map: {world.get_map().name}")

print(f"\nPyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = 0
else:
    DEVICE = 'cpu'


## Sync Mode and Ego Vehicle

Enable synchronous mode, configure Traffic Manager, spawn the ego vehicle, and start the spectator follow helper.


In [ ]:
actors_list = []
walker_controllers = []
npc_vehicles = []

# track spawned actors for cleanup

original_settings = world.get_settings()

settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

traffic_manager = client.get_trafficmanager(8000)
traffic_manager.set_synchronous_mode(True)
traffic_manager.set_global_distance_to_leading_vehicle(2.0)
traffic_manager.set_random_device_seed(42)

print("synchronous mode enabled")
print("traffic manager configured on port 8000")

vehicle_bp = blueprint_library.filter('model3')[0]

spawn_points = world.get_map().get_spawn_points()
spawn_point = random.choice(spawn_points)

vehicle = world.spawn_actor(vehicle_bp, spawn_point)
vehicle.set_autopilot(True)
actors_list.append(vehicle)

world.tick()

print(f"\nEgo vehicle: {vehicle.type_id}")
print(f"Location: ({spawn_point.location.x:.1f}, {spawn_point.location.y:.1f}, {spawn_point.location.z:.1f})")

stop_follow = threading.Event()

def follow_vehicle(veh, w, stop_evt):
    """Continuously positions the CARLA spectator camera behind the ego vehicle."""
    spectator = w.get_spectator()
    while not stop_evt.is_set():
        try:
            if not veh.is_alive:
                break
            t = veh.get_transform()
            loc = t.location + carla.Location(z=3.0) - t.get_forward_vector() * 5.0
            spectator.set_transform(carla.Transform(loc, t.rotation))
        except RuntimeError:
            break
        time.sleep(0.05)

follow_thread = threading.Thread(
    target=follow_vehicle, args=(vehicle, world, stop_follow), daemon=True
)
follow_thread.start()
print("spectator follow thread started")


## Traffic and Pedestrians

Spawn a small scene around the ego vehicle so the detector has nearby road users to observe.


In [ ]:
# add traffic around the ego vehicle

vehicle_bps = blueprint_library.filter('vehicle.*')

npc_spawn_points = spawn_points.copy()
random.shuffle(npc_spawn_points)

for sp in npc_spawn_points[:30]:
    bp = random.choice(vehicle_bps)
    if bp.has_attribute('color'):
        bp.set_attribute('color', random.choice(bp.get_attribute('color').recommended_values))

    npc = world.try_spawn_actor(bp, sp)
    if npc is not None:
        npc.set_autopilot(True, traffic_manager.get_port())

        traffic_manager.vehicle_percentage_speed_difference(npc, random.uniform(-20, 10))
        traffic_manager.auto_lane_change(npc, True)
        traffic_manager.ignore_lights_percentage(npc, 0)

        npc_vehicles.append(npc)
        actors_list.append(npc)

print(f"Spawned {len(npc_vehicles)} NPC vehicles")


walker_bps = blueprint_library.filter('walker.pedestrian.*')
walker_controller_bp = blueprint_library.find('controller.ai.walker')
walkers = []

walker_spawns = []
for _ in range(40):
    loc = world.get_random_location_from_navigation()
    if loc is not None:
        walker_spawns.append(carla.Transform(loc))
    if len(walker_spawns) >= 20:
        break

for wp in walker_spawns:
    bp = random.choice(walker_bps)
    if bp.has_attribute('is_invincible'):
        bp.set_attribute('is_invincible', 'false')
    walker = world.try_spawn_actor(bp, wp)
    if walker is not None:
        walkers.append(walker)
        actors_list.append(walker)

world.tick()
time.sleep(0.5)

for walker in walkers:
    try:
        ctrl = world.spawn_actor(walker_controller_bp, carla.Transform(), attach_to=walker)
        walker_controllers.append(ctrl)
        actors_list.append(ctrl)
    except RuntimeError:
        pass

world.tick()
time.sleep(0.5)

for ctrl in walker_controllers:
    try:
        ctrl.start()
        target = world.get_random_location_from_navigation()
        if target:
            ctrl.go_to_location(target)
            ctrl.set_max_speed(1.0 + random.random() * 1.5)
    except RuntimeError:
        pass

print(f"Spawned {len(walkers)} pedestrians with AI controllers")
print(f"Total actors in scene: {len(actors_list)}")


## RGB Camera

Attach the front camera and store the newest frame as a NumPy array for inference.


In [ ]:
# store the newest camera frame for inference

CAM_WIDTH = 1280
CAM_HEIGHT = 720

latest_frame = {'image': None}

def camera_callback(image):
    """Called by CARLA every simulation tick with a new camera image.

    The image arrives as raw bytes in BGRA format (Blue, Green, Red, Alpha).
    We convert it to a numpy array and drop the alpha channel to get BGR,
    which is what YOLO expects.

    """
    array = np.frombuffer(image.raw_data, dtype=np.uint8)
    array = array.reshape((image.height, image.width, 4))[:, :, :3]
    latest_frame['image'] = array.copy()

camera_bp = blueprint_library.find('sensor.camera.rgb')
camera_bp.set_attribute('image_size_x', str(CAM_WIDTH))
camera_bp.set_attribute('image_size_y', str(CAM_HEIGHT))
camera_bp.set_attribute('fov', '90')

camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4))
camera = world.spawn_actor(camera_bp, camera_transform, attach_to=vehicle)

camera.listen(camera_callback)
actors_list.append(camera)

world.tick()

print(f"Camera: {CAM_WIDTH}x{CAM_HEIGHT}")


## YOLO Model



In [ ]:
# prefer the fine-tuned model when it exists
YOLO_DIR = r'F:\object-detection\Yolo'

finetuned_path = os.path.join(YOLO_DIR, 'runs', 'detect', 'finetune', 'weights', 'best.pt')
baseline_path = os.path.join(YOLO_DIR, 'yolo11s.pt')

if os.path.exists(finetuned_path):
    model_path = finetuned_path
    model_type = 'Fine-tuned (CARLA data)'
else:
    model_path = baseline_path
    model_type = 'Baseline (COCO pre-trained)'

print(f"Model: {model_path}")
print(f"Type:  {model_type}")

model = YOLO(model_path)

dummy = np.zeros((CAM_HEIGHT, CAM_WIDTH, 3), dtype=np.uint8)
t0 = time.time()
_ = model(dummy, verbose=False, device=DEVICE)
warmup_ms = (time.time() - t0) * 1000
print(f"Warm-up: {warmup_ms:.0f}ms")

print(f"\ndriving-related classes:")
driving_classes = ['person', 'bicycle', 'car', 'motorcycle', 'bus',
                   'truck', 'traffic light', 'stop sign']
for name in driving_classes:
    if name in model.names.values():
        cid = [k for k, v in model.names.items() if v == name][0]
        print(f"  [{cid:>2}] {name}")


## Detection Settings

Set confidence, IoU, colors, and video output path for the live loop.


In [ ]:
CONF_THRESHOLD = 0.4
IOU_THRESHOLD = 0.45

# detection settings used by the live loop
CLASS_COLORS = {
    'person':(0, 255, 0),
    'car':(0, 0, 255),
    'truck':(0, 128, 255),
    'bus':(128, 0, 128),
    'motorcycle':(255, 0, 255),
    'bicycle':(255, 165, 0),
    'traffic light':(0, 255, 255),
    'stop sign':(255, 255, 0),
}
DEFAULT_COLOR = (200, 200, 200)

OUTPUT_DIR = r'F:\object-detection\carla_yolo_integration'
output_video_path = os.path.join(OUTPUT_DIR, 'carla_yolo_unified.mp4')


print(f"Color-coded classes: {len(CLASS_COLORS)}")


## Live Detection Loop

Advance CARLA one tick at a time, run YOLO on the latest frame, draw boxes, and save demo.


In [ ]:
# main synchronous detection loop
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(output_video_path, fourcc, 20, (CAM_WIDTH, CAM_HEIGHT))

class_counts = defaultdict(int)
frame_det_counts = []
inference_times = []

print(f"Recording to: {output_video_path}\n")

for _ in range(20):
    world.tick()
time.sleep(0.5)

frame_count = 0
start_time = time.time()

try:
    while True:
        world.tick()

        if not vehicle.is_alive:
            break

        if latest_frame['image'] is None:
            continue

        frame = latest_frame['image'].copy()

        t_infer = time.time()
        results = model(frame, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
                        verbose=False, device=DEVICE)[0]
        inference_ms = (time.time() - t_infer) * 1000
        inference_times.append(inference_ms)

        det_count = 0
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            conf = float(box.conf[0])

            cls_name = model.names[int(box.cls[0])]

            color = CLASS_COLORS.get(cls_name, DEFAULT_COLOR)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            label = f"{cls_name} {conf:.2f}"
            (lbl_w, lbl_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            cv2.rectangle(frame, (x1, y1 - lbl_h - 8), (x1 + lbl_w, y1), color, -1)
            cv2.putText(frame, label, (x1, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

            class_counts[cls_name] += 1
            det_count += 1

        frame_det_counts.append(det_count)

        frame_count += 1
        elapsed = time.time() - start_time
        fps = frame_count / elapsed if elapsed > 0 else 0

        overlay_text = (f"UNIFIED LIVE | FPS: {fps:.1f} | "
                        f"Inference: {inference_ms:.0f}ms | "
                        f"Detections: {det_count}")
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 35), (0, 0, 0), -1)
        cv2.putText(frame, overlay_text, (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        video_writer.write(frame)

        cv2.imshow('Q to quit', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\npressed Q — stopping...")
            break

        if frame_count % 50 == 0:
            avg_inf = np.mean(inference_times[-50:])
            try:
                vel = vehicle.get_velocity()
                speed = 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)
            except RuntimeError:
                speed = 0.0
            print(f"Frame {frame_count:>5} | "
                  f"FPS: {fps:.1f} | "
                  f"Inference: {avg_inf:.0f}ms | "
                  f"Speed: {speed:.1f} km/h | "
                  f"Detections (last 50): {sum(frame_det_counts[-50:])}")

except KeyboardInterrupt:
    print("\nInterrupted by user")

finally:
    cv2.destroyAllWindows()
    video_writer.release()

    total_time = time.time() - start_time
    print(f"Frames processed: {frame_count}")
    print(f"Duration:{total_time:.1f}s")
    if frame_count > 0:
        print(f"Average FPS:{frame_count/total_time:.1f}")
        print(f"Avg inference:{np.mean(inference_times):.0f}ms")
    print(f"Video saved:{output_video_path}")


## Teleport Ego Vehicle

Move the ego vehicle to a new spawn point if the scene gets stuck.


In [ ]:
# move the ego vehicle if it gets stuck
SPAWN_INDEX = None

if SPAWN_INDEX is not None:
    new_spawn = spawn_points[SPAWN_INDEX % len(spawn_points)]
else:
    new_spawn = random.choice(spawn_points)

vehicle.set_autopilot(False)
vehicle.set_transform(new_spawn)
vehicle.set_target_velocity(carla.Vector3D(0, 0, 0))
world.wait_for_tick()
vehicle.set_autopilot(True)

print(f"Teleported to: ({new_spawn.location.x:.1f}, {new_spawn.location.y:.1f}, "
      f"{new_spawn.location.z:.1f})")
print(f"autopilot enabled")

## Restart NPCs

Reissue walker destinations and vehicle autopilot after blocked traffic or collisions.


In [ ]:
# restart scene actors after traffic blocks
restarted_w = 0
for ctrl in walker_controllers:
    try:
        if ctrl.is_alive:
            target = world.get_random_location_from_navigation()
            if target:
                ctrl.go_to_location(target)
                ctrl.set_max_speed(1.0 + random.random() * 1.5)
                restarted_w += 1
    except RuntimeError:
        pass

restarted_v = 0
for npc in npc_vehicles:
    try:
        if npc.is_alive:
            npc.set_autopilot(True, traffic_manager.get_port())
            restarted_v += 1
    except RuntimeError:
        pass

print(f"Pedestrians restarted:{restarted_w}/{len(walker_controllers)}")
print(f"Vehicles restarted:{restarted_v}/{len(npc_vehicles)}")


## Cleanup

Stop sensors, restore async mode, and destroy spawned actors.


In [ ]:
try:
    stop_follow.set()
    follow_thread.join(timeout=2.0)
    print("Follow thread stopped")
except NameError:
    pass

print(f"Cleaning up {len(actors_list)} actors...")

# restore async mode before destroying actors

try:
    world.apply_settings(original_settings)
    print("restored asynchronous mode")
    time.sleep(0.5)
except Exception:
    print("Could not restore settings")

sensor_count = 0
for actor in actors_list:
    try:
        if actor is not None and actor.is_alive and actor.type_id.startswith('sensor'):
            actor.stop()
            sensor_count += 1
    except RuntimeError:
        pass
print(f"  Stopped {sensor_count} sensor(s)")
time.sleep(0.3)

ctrl_count = 0
for ctrl in walker_controllers:
    try:
        if ctrl is not None and ctrl.is_alive:
            ctrl.stop()
            ctrl_count += 1
    except RuntimeError:
        pass
print(f"Stopped {ctrl_count} walker controller(s)")
time.sleep(0.3)

destroyed = 0
batch_size = 5
reversed_list = list(reversed(actors_list))

for i in range(0, len(reversed_list), batch_size):
    batch = reversed_list[i:i + batch_size]
    for actor in batch:
        try:
            if actor is not None and actor.is_alive:
                actor.destroy()
                destroyed += 1
        except RuntimeError:
            pass
    time.sleep(0.3)

print(f"Destroyed {destroyed} actor(s)")

actors_list.clear()
walker_controllers.clear()
npc_vehicles.clear()

try:
    traffic_manager = client.get_trafficmanager(8000)
    traffic_manager.set_synchronous_mode(False)
except Exception:
    pass

print("\nCARLA restored to asynchronous mode")
